# 00 - The statistics this project uses

**Purpose.** To make every statistical move in this repository readable, using this camera's own
frames as the worked example. Thirteen ideas, in the order the project needs them, each one
demonstrated on real pixels rather than stated. It is written to be read slowly: every term is
defined before it is used, and no section leans on one that has not happened yet.

It opens with a section of **physics rather than statistics** - the chain of physical steps that
turns a photon into a number in a FITS file. That section is here because the rest of the
notebook is about numbers produced by that chain, and knowing where in it each effect enters is
what turns the rules that follow into reasons. Read `CLAUDE.md`'s spec-sheets rule as it applies
here: the chain is a *hypothesis that organises our measurements*, and where session 01's frames
can test a claim about it, section 0 runs the test.

**What it is not for.** It is not a measurement, it writes nothing to `results/`, and it
introduces no threshold. Where a number appears it is illustrative, and the notebook says so.
It is also not a statistics course: the test for including an idea was "does a later notebook
lean on it", and things this project does not do - hypothesis tests, p-values, Bayesian
inference, regression beyond a straight line - are deliberately absent. Nor is it a sensor
engineering text: section 0 goes exactly as deep as is needed to explain the later sections and
then stops.

**Read it once, then stop reading it.** Every later explainer notebook cites this one for
vocabulary instead of re-deriving it. That is the point.

**It needs `data/session01/frames/`** - the bias sweep, 2,790 frames, 5.5 GB on `C:`. Those
frames are disposable by policy, so if they have been cleared this notebook stops at the first
read. It does not fall back to simulated data: fabricated pixels in a repository this careful
about provenance would be a bad habit bought cheaply.

## The one-sentence version of each section

### Part I - what a number from this camera actually is

| # | idea | where the project uses it |
|---|---|---|
| 0 | the chain from photon to FITS value, and where each effect enters it | why `R` rises with gain, why HCG exists, what `offset` does |
| 1 | a pixel value is a random variable | everything |
| 2 | mean and standard deviation | `pedestal`, `R` |
| 3 | variances add, sigmas do not | the `sqrt(2)` in every pair difference |

### Part II - measuring the width

| # | idea | where the project uses it |
|---|---|---|
| 4 | spread across pixels vs spread across time | `fpn_ratio`, and why a master bias may be unnecessary |
| 5 | why we difference two frames | `R` from a bias pair (L10) |
| 6 | median and MAD, and what "robust" buys | `R_sd` vs `R_mad` in `bias_sweep.csv` |

### Part III - how well we know the width

| # | idea | where the project uses it |
|---|---|---|
| 7 | the standard error of a mean | the `uncertainty` field on every constant |
| 8 | quantisation | why low-gain noise is partly the grid, and L14's zero dark |
| 9 | the Gaussian tail | "15 read noises above the floor" |
| 10 | fractions, not extremes | L13, and the offset decision |

### Part IV - turning numbers into decisions

| # | idea | where the project uses it |
|---|---|---|
| 11 | fitting a line, and reading residuals | `pedestal_fit`, and the branch split |
| 12 | a slope against its own uncertainty | the drift trace, and whether session 02 interleaves |

In [ ]:
import math, pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import fits as F, spatial as SP, stats as ST

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

FRAMES = pathlib.Path("..") / "data" / "session01" / "frames"
RESULTS = pathlib.Path("..") / "results"

if not FRAMES.is_dir():
    raise SystemExit(
        f"{FRAMES} is missing.  This notebook reads session 01's own frames and has\n"
        "no simulated fallback.  Re-run the capture half of 03_bias_sweep.ipynb "
        "(about an hour, cooling included) or read this notebook without executing it.")


def plane(name, which="G1"):
    '''One CFA plane of one frame, in ADC counts.  `name` is a file stem.'''
    mosaic, _ = F.read(FRAMES / f"{name}.fits")
    return ST.to_adc(SP.split(mosaic)[which]).astype(np.float64)


# The workhorse: 20 frames at gain 100, offset 15, all captured back to back at
# -10 C.  Nothing about the camera changed between them, so every difference
# below is noise and nothing else.
STACK = np.stack([plane(f"coarse_g100_o015_{i:03d}") for i in range(20)])
NPX = STACK[0].size
print(f"{STACK.shape[0]} frames, each {STACK.shape[1]} x {STACK.shape[2]} pixels "
      f"of one plane = {NPX:,} values per frame")
print("(the ROI is 1024 x 1024 of the mosaic; splitting RGGB gives four planes of "
      "512 x 512)")

SWEEP = pd.read_csv(RESULTS / "bias_sweep.csv")
print(f"\nand {len(SWEEP)} rows of published sweep results, for the sections that read "
      "them back")

---

# Part I - what a number from this camera actually is

## 0. The chain: from a photon to a number in a file

Every other section of this notebook is about numbers. This one is about where those numbers come
from, because two of the quantities the project measures - the **pedestal** and the **read
noise** - enter at specific and *different* places in the process, and almost everything that
looks surprising about them later is explained by which place.

### The words, before any of them are used

- **Photon.** One particle of light. The sensor counts them, one at a time.
- **Pixel.** One light-collecting bucket on the sensor. This camera has 3840 x 2160 of them,
  each 2.9 micrometres across.
- **Electron.** When a photon is absorbed in a pixel it knocks one electron loose. The electrons
  pile up and stay put for the duration of the exposure. **That pile of electrons is the real
  measurement** - everything after this point is an attempt to find out how big it is.
- **Exposure.** The interval during which the pile is allowed to accumulate. `t` in the model.
- **Read out.** Everything the camera does *after* the exposure to turn each pile of electrons
  into a number. It is not one step; it is the chain below, and it is where this section lives.
- **ADC** - analogue-to-digital converter. The component that takes a continuous voltage and
  picks the nearest value from a fixed ladder of allowed values. This camera's ADC has 12 bits:
  4096 rungs, numbered 0 to 4095.
- **ADC count.** One rung of that ladder. It is this project's one unit, and `CLAUDE.md` defines
  it; every number in `results/` is in these.
- **Gain.** A dial on the amplifier, 0 to 600 on this camera. It does *not* make the sensor
  collect more light. It changes how large a number a given pile of electrons produces.
- **Pedestal.** A constant added to every pixel so that the numbers never sit at zero. About 68.6
  counts at gain 100, offset 15 on this rig. It is the same in every frame, which - as section 1
  will make precise - is exactly what stops it being noise.
- **Read noise.** The random, unrepeatable part that the readout chain itself contributes. The
  project calls it `R`, and measuring it as a function of gain was the whole of session 01.

### The chain

```
 [1] photodiode               electrons accumulate during the exposure
      |
 [2] transfer to the          charge becomes a VOLTAGE:  V = Q / C
      floating diffusion      (the "conversion gain", in microvolts per electron)
      |
 [3] in-pixel source          buffers that voltage onto the column wire
      follower
      |
 [4] column amplifier         <-- the ZWO "Gain" dial acts HERE
      |
 [5] column ADC               voltage becomes an INTEGER, 0..4095
      |
 [6] digital block            offset added; x16 shift; white balance
      |
      the FITS file we read
```

**Stage 1 - the photodiode.** Photons in, electrons out. No electronics have touched the
measurement yet, but it is *already* random: photons arrive at random times, so even a perfectly
steady source delivers a pile whose size varies from exposure to exposure. That is **shot
noise**, and it is a property of light, not of the camera. Dark current lives here too - heat
shakes electrons loose whether or not any light arrives. Both grow with exposure time, which is
the `(F_obj + F_sky + D) * t` half of the model in `MISSION.md`.

**Stage 2 - the floating diffusion.** The pile of charge is moved onto a very small capacitor,
and charge on a capacitor is a voltage: `V = Q / C`. This is the moment electrons become volts,
and the exchange rate is `1 / C`, measured in microvolts per electron. It is called the
**conversion gain**, and it is a property of the silicon rather than a setting.

Except that on this sensor it is partly a setting. Some Sony sensors provide a switchable extra
capacitor - **dual conversion gain**. Switch the extra capacitance out, `C` falls, and every
electron produces *more* volts. **This camera does that at gain 200**, and it is the reason
`read_noise_at_hcg` is 1.0768 counts while gain 100 measures 1.4251: the dial went up and the
noise went *down*. HCG is not amplification. It is a better exchange rate at the source, and
section 6 and section 11 both see it as a break in an otherwise smooth curve.

**Stage 3 - the source follower.** A single small transistor that copies the floating-diffusion
voltage onto a long wire running the length of the column. It is real silicon at a real
temperature, and it wobbles. This is the largest single contributor to read noise.

**Stage 4 - the column amplifier.** A programmable amplifier that multiplies the voltage before
it reaches the ADC. **The `Gain` control acts here.** Everything upstream of this point gets
multiplied by it; everything downstream does not. That sentence is the whole of the second half
of this section.

**Stage 5 - the ADC.** The voltage becomes one of 4096 integers. Two things happen: the
comparator circuitry has noise of its own, and the rounding to the nearest rung is itself an
error. The rounding is **quantisation**, and section 8 is about what it does to a measurement.

**Stage 6 - the digital block.** Integer arithmetic, after all physics is finished. The black
level offset is added here, the value is shifted x16 into its 16-bit container, and the white
balance multipliers are applied - the ones `asi.neutralise_white_balance` exists to switch off,
for the reason `CLAUDE.md` gives.

### Where the pedestal enters, and how we know

There is a decisive test available in session 01's own data, and it needs no new frames. The
argument is one sentence:

> **Anything injected before stage 4 is multiplied by stage 4. Anything injected after it is
> not.**

So: sweep the `Offset` control at several gains and ask how many ADC counts one offset unit is
worth. If the answer changes with gain, the offset is analogue and upstream. If the answer is the
same at every gain, it is applied after the amplifier.

The sweep covers offsets 0 to 50 in steps of 5 at every gain, so the question is already
answered; the cell below fits a straight line through each gain's eleven offsets. Two numbers
come out of each fit - a **slope** in counts per offset unit, and an **intercept**, which is what
the pedestal would be with the offset control set to zero. (The machinery of fitting a line is
section 11's subject; here only the two numbers are needed, and `np.polyfit` supplies them.)

The result is that the slope is **4.000 counts per offset unit at every gain**, unmoved in the
fourth decimal place across amplifier settings that differ by a factor of a thousand, while the
intercept climbs from 2.7 counts to 978. So the pedestal is **two things added in two places**:

```
pedestal(gain, offset)  =  A(gain)          +   4.000 * offset
                           |                    |
                           analogue baseline,   digital, at stage 6,
                           amplified by the     untouched by the gain
                           gain at stage 4
```

A second test confirms it independently. If the offset were injected as an analogue voltage, it
would come from a real circuit, and real circuits have noise - turning the offset up to 50 should
measurably widen the frame. The `R_at_offset` column says it does not, at any gain. A noiseless
addition is what integer arithmetic looks like.

One honest limit on the conclusion: this proves the offset is applied **after the gain stage** and
**adds no measurable noise**. It cannot by itself separate "an integer added after the ADC" from
"a perfectly noiseless analogue offset injected at the ADC's own reference". Both are downstream
of stage 4, which is all that any later section needs. That the exact figure is 4.000 and not
3.97 is what points at the digital answer.

In [ ]:
# Does one offset unit buy the same number of counts at every gain?
grid = SWEEP.pivot_table(index="gain", columns="offset", values="pedestal").dropna()
off = np.asarray(grid.columns, dtype=float)

print("fitting  pedestal = intercept + slope * offset,  separately at each gain")
print(f"\n{'gain':>5} {'slope':>10} {'intercept':>11} {'worst resid':>12}   "
      f"{'amplification':>13}")
for g, row in grid.iterrows():
    y = row.to_numpy()
    slope, intercept = np.polyfit(off, y, 1)
    worst = np.abs(y - (slope * off + intercept)).max()
    print(f"{g:>5.0f} {slope:>10.4f} {intercept:>11.3f} {worst:>12.3f}   "
          f"{10 ** (g / 200.0):>13.1f}x")

print("\nThe slope column is the answer: 4.000 counts per offset unit, everywhere, while the\n"
      "amplification spans 1x to 1000x.  The offset is applied AFTER the gain stage.\n"
      "The intercept column is the other half of the pedestal -- the analogue baseline, which\n"
      "the gain does amplify.  Its shape, including the drop between gain 190 and 200 where\n"
      "the sensor switches conversion gain, is section 11's worked example.")

# And does turning the offset up add any noise of its own?
noise = SWEEP.pivot_table(index="gain", columns="offset", values="R_at_offset").dropna()
print(f"\n{'gain':>5}  R at offset 5   R at offset 50    change     R at offset 0")
for g, row in noise.iterrows():
    a, b, z = row[5], row[50], row[0]
    print(f"{g:>5.0f} {a:>13.4f} {b:>15.4f} {b / a - 1:>+9.2%} {z:>16.4f}")
print("\nFlat to a fraction of a percent: the offset contributes no noise.  The one systematic\n"
      "is the last column -- offset 0 reads consistently LOW, because there the pedestal sits\n"
      "a couple of counts above zero and the noise's lower tail is being cut off by the floor.\n"
      "That is not a quieter camera; it is a truncated distribution, and sections 9 and 10 are\n"
      "about how the project decided offset 0 was unusable.")

### Where the read noise enters

The pedestal had an answer of the form "at stage 6". Read noise does not, and the difference is
the interesting part.

**Read noise is not added at a stage. It accumulates along the whole chain.** Every active
component between the floating diffusion and the file contributes something, and - by a rule that
section 3 will establish properly - those contributions combine by adding their *squares*:

```
R_total^2  =  R_stage2^2 + R_stage3^2 + R_stage5^2 + ...
```

So the useful question is not "which stage" but **"which side of stage 4"**, because stage 4 is
the multiplier.

**Upstream of the gain.** The dominant term is the source follower at stage 3, and it brings
three flavours worth knowing by name:

- *thermal noise* - the random motion of charge carriers, present at any temperature above
  absolute zero;
- *1/f*, or flicker noise - slow drift, from charge being trapped and released in the
  transistor's oxide;
- *RTS*, random telegraph signal - a single trap flipping one pixel between two discrete levels,
  frame to frame. These are the "telegraph pixels" that section 6 meets as outliers and section
  10 meets again as a rival explanation for pixels reading zero.

Stage 2 contributes **reset noise** (also called kTC noise), from the randomness in resetting the
capacitor's charge before the transfer. Most of it is removed on the chip by **correlated double
sampling**: the floating diffusion is read *twice*, once immediately after the reset and once
after the charge arrives, and the two are subtracted. Whatever the reset happened to leave
behind is present in both reads and cancels.

That trick is worth pausing on, because section 5 is the same idea at a completely different
scale: **subtract two reads to remove everything they have in common.** The chip does it in
nanoseconds to kill reset noise; we do it between two adjacent bias frames to kill fixed pattern.

All of these are upstream, so **the gain multiplies them**, exactly as it multiplies the signal.
Expressed in electrons - "referred to the input" - they are a fixed quantity that no dial can
change.

**Downstream of the gain.** The ADC's comparator noise, and quantisation itself, which
contributes about `1 / sqrt(12)` = 0.29 counts no matter what. These are **fixed in ADC counts**,
because they happen after the multiplier. Referred back to the input, in electrons, they shrink
as the gain rises - each count now stands for fewer electrons.

### Why that explains the shape of `R(gain)`

Put the two families in one expression:

```
R(gain)^2  in counts  =  [ R_upstream(electrons) * gain ]^2  +  [ R_downstream(counts) ]^2
                          \_ grows with gain _/                  \_ flat with gain _/
```

and then read the measured column against it:

| gain | measured `R`, counts | which term is in charge |
|---|---|---|
| 0 | 0.66 | downstream. The upstream noise is *smaller than one rung* and cannot be seen at all |
| 100 | 1.43 | the crossover |
| 200 | 1.08 | HCG: stage 2 changed, so the upstream term fell in counts |
| 300 | 2.90 | upstream, amplified |
| 600 | 76.90 | upstream, amplified hugely |

The first row is the one that misleads people. `R = 0.66` counts at gain 0 is not a quiet camera;
the camera's own noise is *invisible* there, buried under the ADC's rounding. Section 8 is about
that situation and section 6 shows a statistic being destroyed by it.

It is also, finally, why the gain dial is worth having. Raising the gain does not reduce the
upstream noise - it **outruns the downstream noise**, until the fixed ADC contribution stops
mattering. So `R` measured *in electrons* falls as gain rises and then flattens onto the upstream
floor. HCG at gain 200 is a free step down that curve: a better exchange rate at stage 2 shrinks
everything downstream in relative terms while amplifying nothing.

This notebook cannot plot that curve, because converting counts to electrons needs `g(gain)` from
a photon transfer curve, and `MISSION.md` still lists it as unmeasured. Everything here stays in
ADC counts, deliberately.

### The section in one table

| what | enters at | multiplied by the gain? | the evidence in this repo |
|---|---|---|---|
| shot noise, dark current | stage 1 | yes | scales with `t`; sections 3 and 5 |
| reset (kTC) noise | stage 2 | mostly cancelled on-chip by CDS | not separable from our data |
| conversion gain, and the HCG switch | stage 2 | it *is* the exchange rate | `R` drops 3.30 -> 1.08 at gain 200 |
| source follower: thermal, 1/f, RTS | stage 3 | **yes** | `R` grows with gain above the crossover |
| ADC and quantisation noise | stage 5 | **no** | `R` floors at 0.66; `R_mad` pinned at 1.0484 |
| analogue baseline `A(gain)` | before stage 4 | **yes** | fitted intercept, 2.7 -> 978 counts |
| the `Offset` control | stage 6 | **no** | slope exactly 4.000 counts/unit at every gain |

With that in hand, the rest of the notebook is about what to *do* with the numbers the chain
produces.

---

## 1. A pixel value is a random variable

Point the camera at nothing, cover it, set the shortest exposure the camera allows, and read
pixel (500, 500) twenty times. The pile of electrons is empty every time, and identical every
time. You still do not get the same number twenty times. You get twenty numbers scattered around
a centre, and section 0 says where the scatter was manufactured: stages 2 through 5, on the way
out.

That is the foundation of everything here. **A pixel value is not a measurement with an error
attached - it *is* a draw from a distribution.**

### The two words

A **distribution** is the full description of what values a quantity can take and how often. You
never see one directly; you see draws from it. Two features of a distribution matter for this
project, and it is worth being strict about keeping them separate, because they behave in
opposite ways.

- The **centre** - roughly, where the values pile up. When the camera is reading darkness, the
  centre is the pedestal.
- The **width** - how far the values scatter either side of the centre. That is what we call
  noise, and when the camera is reading darkness it is `R`.

### The two things that follow, and why the bench nights are what they are

- **The centre can be found as precisely as you like, by averaging enough draws.** There is no
  floor. This is why session 01 can quote a pedestal to a fraction of a count using frames whose
  individual pixels wobble by whole counts, and section 7 is the arithmetic of exactly how
  precisely.
- **The width cannot be averaged away for a single frame.** More draws let you *know* the width
  more precisely; they do not make it smaller. The width is a property of the sensor and the
  electronics - section 0 named the components responsible - and the only ways to beat it are to
  collect more signal, so that the width matters less in proportion, or to combine more frames,
  which section 3 prices.

That asymmetry is easy to lose, because both are "measured better with more frames". The centre
gets better *known* and the width gets better *known*; only the centre's uncertainty goes to
zero.

The first panel below is one pixel, twenty times. The second is the same idea for a hundred
pixels at once, each one centred on its own mean first, to show that the shape is not a fluke of
the pixel we happened to pick.

In [ ]:
one = STACK[:, 500, 500]
print("pixel (500, 500) across 20 frames, ADC counts:")
print(np.round(one, 2))
print(f"\ncentre (mean) {one.mean():.3f}    width (sd) {one.std(ddof=1):.3f}")
print("\nNothing about the camera or the scene changed between those twenty reads.  The\n"
      "spread is the readout chain, and nothing else.")

fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
ax[0].hist(one, bins=np.arange(one.min() - 0.5, one.max() + 1.5, 1.0),
           color="0.4", edgecolor="white")
ax[0].axvline(one.mean(), color="crimson", lw=1)
ax[0].set(title="one pixel, 20 frames", xlabel="ADC counts", ylabel="frames")

rng = np.random.default_rng(0)
ys, xs = rng.integers(0, STACK.shape[1], 100), rng.integers(0, STACK.shape[2], 100)
many = STACK[:, ys, xs]                       # 20 frames x 100 pixels
ax[1].hist((many - many.mean(axis=0)).ravel(), bins=40, color="0.4", edgecolor="white")
ax[1].set(title="100 pixels, each centred on its own mean",
          xlabel="ADC counts from that pixel's centre", ylabel="observations")
fig.tight_layout()

## 2. Mean and standard deviation

Section 1 said a distribution has a centre and a width. This section is how each is computed from
a set of draws, and why the recipe for the width is the odd shape it is.

### The mean

The **mean** is the centre: add the values, divide by how many there are. There is nothing else
to it, and it is the estimator this project uses for every centre it publishes.

### The standard deviation

The **standard deviation** - written `sd`, or the Greek letter sigma - is the width. The recipe
is four steps:

1. subtract the mean from each value, giving that value's **deviation**;
2. **square** every deviation;
3. average the squares;
4. take the square root of that average.

Step 3's result, before the square root, has its own name: the **variance**. So `variance = sd
squared`, and equivalently `sd = sqrt(variance)`.

The obvious objection is step 2. Why square, when the plain distance from the centre is a
perfectly good measure of how far away something is? The average of the absolute deviations is a
real quantity, it is easier to explain, and on this frame it comes out to about 0.8 times the
`sd` - so it is not even very different.

The answer is the whole of section 3, and it is worth stating here as a promise rather than
leaving it as a mystery: **squared deviations add up when independent noise sources combine, and
plain distances do not.** Sigma is defined the way it is precisely so that combining noise is
arithmetic instead of guesswork. The average absolute deviation is a fine description of one
distribution and useless for combining two.

Which leads to a habit worth adopting early, and one this repository follows without exception:

> **Quote `sd`, because it is in the same units as the pixels. Compute with variance, because it
> is the one that adds.**

### One piece of code vocabulary

`ddof` is the choice between dividing by `n` or by `n - 1` at step 3. The `n - 1` version corrects
a small bias that appears when you estimate the width using a mean you also estimated from the
same data. On a million pixels the difference is one part in a million and nothing turns on it.
It matters in exactly one place in this project: at the PixInsight boundary, where PI uses `n - 1`
and numpy defaults to `n` (L22). **This project sets `ddof=1` only there.**

In [ ]:
x = STACK[0]
dev = x - x.mean()

print("one frame, one plane, gain 100 -- the four steps of the recipe")
print(f"  1. mean                  {x.mean():.4f} counts")
print(f"  2. squared deviations    (dev ** 2), one per pixel")
print(f"  3. their average         {(dev ** 2).mean():.4f} counts^2   <- the VARIANCE")
print(f"  4. its square root       {np.sqrt((dev ** 2).mean()):.4f} counts   <- the SD")
print(f"\n  numpy's std()            {x.std():.4f} counts   (the same thing, in one call)")
print(f"  variance from the sd     {x.std() ** 2:.4f} counts^2  (sd ** 2, as promised)")

print(f"\nthe 'obvious' alternative width:")
print(f"  mean |deviation|         {np.abs(dev).mean():.4f} counts   "
      f"({np.abs(dev).mean() / x.std():.2f} x the sd)")
print("  -- a perfectly good description of this one distribution, and the wrong tool the\n"
      "     moment two noise sources have to be combined.  Section 3.")

print(f"\nddof, on {x.size:,} pixels: {x.std():.6f} (ddof=0) vs {x.std(ddof=1):.6f} (ddof=1)")
print(f"  they differ by {abs(x.std(ddof=1) / x.std() - 1):.2e} -- which is why it only ever\n"
      "  matters at the PixInsight boundary, and nowhere else in this repo.")

## 3. Variances add; sigmas do not

This is the promise from section 2, and it is the single most load-bearing fact in the notebook.
Every `sqrt` in this repository is this fact wearing a different hat.

### The rule

Two noise sources are **independent** if knowing what one did on a given read tells you nothing
about what the other did. The readout chain's contributions are independent of each other in
this sense: the ADC's rounding error has no idea what the source follower just did.

When two independent sources of width `a` and `b` combine, the result has width

```
sqrt(a^2 + b^2)      and NOT      a + b
```

Equivalently, and more usefully: **their variances add.** `var_total = var_a + var_b`. That is
the sentence to remember, because it is the one that generalises - three sources, twenty frames,
a whole chain of stages.

The intuition, if you want one: two independent wobbles are as likely to cancel as to reinforce,
so they do not accumulate as fast as simple addition would suggest. The squares are the
bookkeeping that gets this exactly right.

### Three consequences, all of which the project runs on

**Small noise sources almost vanish beside large ones.** Add a 0.3-count source to a 1.0-count
source and the answer is `sqrt(1.0^2 + 0.3^2)` = 1.044, not 1.3. A source three times smaller
contributes four percent. This is why the read-noise question is only interesting when the other
terms are small - which is exactly the short-sub-exposure case the whole model is about, and why
section 0's downstream noise stops mattering once the gain has outrun it.

**Subtracting two frames does not cancel their noise - it adds it.** The *signal* cancels,
because it is the same in both. The *noise* is different in each, so it combines by the rule
above: two frames of width `R` give a difference of width `R * sqrt(2)`. Hence, everywhere in
this repo:

```
R = sd(frame2 - frame1) / sqrt(2)
```

That is section 5's measurement, and the `sqrt(2)` is not a fudge factor - it is this rule.

**Averaging `N` frames divides the noise by `sqrt(N)`, not by `N`.** Four frames buy a factor of
two; a hundred buy a factor of ten. Stacking is therefore expensive, and it is why `eta_comb` -
how close a real stack gets to that ideal, once registration and rejection have had their say -
is worth measuring rather than assuming. That measurement is `05`'s job.

The cell below checks all three: the first on synthetic draws where the true answer is known, the
second and third on session 01's own frames.

In [ ]:
# 1. the rule itself, on draws where we know the right answer
rng = np.random.default_rng(1)
a, b = rng.normal(0, 1.0, 400_000), rng.normal(0, 0.3, 400_000)
print(f"a alone {a.std():.4f}   b alone {b.std():.4f}")
print(f"a + b   {(a + b).std():.4f}   predicted sqrt(1.0^2 + 0.3^2) = "
      f"{math.hypot(1.0, 0.3):.4f}   (and not 1.3)")
print(f"in variance: {a.var():.4f} + {b.var():.4f} = {a.var() + b.var():.4f}, "
      f"measured {(a + b).var():.4f}")

# 2. differencing adds noise -- on two real bias frames, same setting, back to back
d = STACK[1] - STACK[0]
print(f"\ntwo real bias frames:")
print(f"  frame 0 sd    {STACK[0].std():.4f}")
print(f"  frame 1 sd    {STACK[1].std():.4f}")
print(f"  difference sd {d.std():.4f}   = {d.std() / STACK[0].std():.4f} x one frame "
      f"(sqrt(2) = {math.sqrt(2):.4f})")
print(f"  so R = sd(difference) / sqrt(2) = {d.std() / math.sqrt(2):.4f} counts")

# 3. averaging beats noise as sqrt(N), and no faster
n = np.arange(1, 21)
got = np.array([STACK[:k].mean(axis=0).std() for k in n])
fig, ax = plt.subplots(figsize=(4.2, 2.6))
ax.plot(n, got, "o-", ms=3, color="0.3", label="measured")
ax.plot(n, got[0] / np.sqrt(n), "--", color="crimson", label="sd / sqrt(N)")
ax.set(xlabel="frames averaged", ylabel="sd of the average, counts",
       title="averaging beats noise slowly")
ax.legend(); fig.tight_layout()
print(f"\n20 frames averaged: {got[0]:.4f} -> {got[-1]:.4f} counts, a factor of "
      f"{got[0] / got[-1]:.2f} for 20x the data (sqrt(20) = {math.sqrt(20):.2f}).")
print("The measured factor falls a little short of the ideal, and the shortfall is the\n"
      "point: averaging removes what differs between frames and leaves whatever is the\n"
      "same in all of them.  On this sensor that residue is small.  Measuring it is\n"
      "section 4, and it is the reason a master bias may not be needed here at all.")

---

# Part II - measuring the width

## 4. Two directions of spread, and why the project always says which one

Take the same stack of twenty frames. There are two completely different questions you can ask
about its spread, they produce different numbers, and confusing them is the most expensive
mistake available in this subject.

- **Across pixels, within one frame** - call it *spatial*. Take one frame; how much do its
  quarter-million pixels differ from each other? This mixes the noise with any **fixed
  pattern**: pixels that are reliably a little brighter or darker than their neighbours, frame
  after frame, because of small permanent differences in the silicon.
- **Across frames, at one pixel** - call it *temporal*. Follow a single pixel through the stack;
  how much does it wobble? This is noise only. A fixed offset that is always there contributes
  nothing to the wobble, because it does not change.

Section 1's twenty draws of pixel (500, 500) were a temporal measurement. A single frame's `sd`
in section 2 was a spatial one.

### The ratio is the diagnostic

If the spatial spread is much wider than the temporal one, the sensor has structure - and
structure that is the same every frame is structure a **master bias** could subtract away. If
the two are equal, there is no structure to remove and a single number is the whole bias model.

This camera measures `fpn_ratio = 1.011` averaged over 77 gains, and 1.0016 at the gain 100 used
here. **That is why a scalar pedestal is the entire bias model this project needs** - a finding,
and one that saves a great deal of work downstream.

### Two subtleties, both of which bite

**Combine per-pixel widths through the variances, not the sigmas.** Section 3's rule again:
average the squares, then take the square root. Averaging the sigmas directly underestimates - by
3.6% on this stack - for the same reason the average of two square roots is not the square root
of the average.

**The temporal spread comes out slightly *larger* than the spatial one**, which looks impossible
until you notice what else is moving. The whole frame breathes: each frame's mean wanders by about
0.26 counts from the next, and that wobble is shared by every pixel at once. It lands in the
per-pixel temporal spread, and it *cannot* land in a single frame's spatial spread, because
within one frame it is a constant - and a constant contributes nothing to a spread. Subtract it,
in variance because variances add, and the temporal number reproduces the pair-difference `R` to
four digits.

That is not an aside. It is the reason the project measures `R` from **adjacent** pairs: two
frames a second apart share almost all of that wobble and it cancels in the difference, while
twenty frames spanning a minute do not.

The map in the cell below makes the fixed-pattern point visually. The per-pixel *mean* of 20
frames averages the noise down by `sqrt(20)` = 4.5 and would leave any fixed pattern standing in
plain sight.

In [ ]:
spatial = STACK[0].std()                                  # one frame, across its pixels
temporal = math.sqrt(STACK.var(axis=0, ddof=1).mean())    # per pixel, across frames
naive = STACK.std(axis=0, ddof=1).mean()                  # the tempting wrong way

print(f"spatial  sd  (1 frame, {NPX:,} pixels)    {spatial:.4f} counts")
print(f"temporal sd  ({NPX:,} pixels, 20 frames) {temporal:.4f} counts")
print(f"(averaging the sigmas instead of the variances: {naive:.4f}, "
      f"{naive / temporal - 1:+.1%} low)")

# The whole frame breathes.  That wobble is in the temporal number and cannot be
# in the spatial one, so take it out -- in variance, because variances add.
wobble = STACK.mean(axis=(1, 2)).std(ddof=1)
temporal_fixed = math.sqrt(STACK.var(axis=0, ddof=1).mean() - wobble ** 2)
R_pair = math.sqrt(np.mean([(STACK[i + 1] - STACK[i]).var() / 2 for i in range(0, 20, 2)]))

print(f"\nframe-to-frame wobble of the whole plane   {wobble:.4f} counts")
print(f"temporal sd with the wobble removed        {temporal_fixed:.4f} counts")
print(f"R from adjacent pairs (section 5)          {R_pair:.4f} counts   <- they agree")
print(f"\nfixed-pattern ratio, spatial / R           {spatial / R_pair:.4f}")
print("At 1.0 there is no fixed pattern to remove.  `bias_sweep.csv` gives 1.0016 for this\n"
      "gain and 1.011 averaged over all 77 -- a scalar pedestal is the whole bias model.")

fig, ax = plt.subplots(1, 3, figsize=(10, 2.8))
crop = slice(400, 600), slice(400, 600)
m = STACK[0][crop]
ax[0].imshow(m, cmap="gray", vmin=m.mean() - 3 * spatial, vmax=m.mean() + 3 * spatial)
ax[0].set(title=f"one frame (sd {spatial:.3f})")

avg = STACK.mean(axis=0)[crop]
ax[1].imshow(avg, cmap="gray", vmin=m.mean() - 3 * spatial, vmax=m.mean() + 3 * spatial)
ax[1].set(title=f"mean of 20 (sd {STACK.mean(axis=0).std():.3f})")

ax[2].hist(STACK[0].ravel(), bins=60, histtype="step", label="one frame", color="0.3")
ax[2].hist(STACK.mean(axis=0).ravel(), bins=60, histtype="step", label="mean of 20",
           color="crimson")
ax[2].legend(); ax[2].set(xlabel="ADC counts", title="the same two, as histograms")
for a in ax[:2]:
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout()
print("\nThe middle panel is 4.5x quieter and shows no surviving structure.  A sensor with\n"
      "fixed pattern would look smooth in panel 1 and *mottled* in panel 2, because the\n"
      "averaging removes the noise and leaves the pattern.")

## 5. Why read noise is measured from a *difference* of two frames

Section 4 said one frame's spread mixes noise with fixed pattern. So measuring read noise from a
single frame's `sd` overstates it by whatever pattern is present - and you cannot tell by how
much, because both live in the same number.

Differencing two frames taken back to back fixes this exactly. Anything the same in both -
the pedestal, hot pixels, column structure, any fixed offset whatsoever - **subtracts to zero**.
What survives is the part that was different, which is the noise, at `sqrt(2)` times its
single-frame width.

This is not a small refinement. The alternative offered by the textbook - read noise as the
*intercept* of a photon transfer curve - extrapolates a line from bright exposures back to zero
signal, and L10 records that trick returning **7.1 e- for a true 3.0 e-**. The pair difference
needs no fit, no extrapolation, and no assumption about linearity.

The one thing it demands is that the two frames be **adjacent in time**: anything that drifts
between them stops cancelling and starts contributing. That is precisely what section 12's drift
trace was shot to check.

In [ ]:
single = STACK[0].std()
pair = (STACK[1] - STACK[0]).std() / math.sqrt(2)
master = STACK[2:].mean(axis=0)                 # a "master bias" from 18 frames
vs_master = (STACK[0] - master).std()

print(f"one frame's sd                        {single:.4f}  (noise + any fixed pattern)")
print(f"pair difference / sqrt(2)             {pair:.4f}  (noise alone -- this is R)")
print(f"frame minus an 18-frame master        {vs_master:.4f}  (noise + the master's own noise)")
print(f"  the master carries sd/sqrt(18) of its own: predicted "
      f"{math.hypot(pair, pair / math.sqrt(18)):.4f}")
print(f"\nOn this sensor all three nearly agree, because fpn_ratio is 1.01 -- there is no\n"
      f"fixed pattern for the difference to remove.  That is a *finding*, not a licence:\n"
      f"on a sensor with structure the first number would be visibly the largest, and the\n"
      f"pair difference is the only one of the three that is right either way.")

## 6. Median and MAD: what "robust" means, and what it costs

The mean and `sd` are exquisitely sensitive to a handful of extreme values, because squaring a
large deviation makes it enormous. One pixel 1000 counts off the centre contributes as much to
the variance as a million pixels one count off.

The robust replacements:

- the **median** - the middle value when sorted - replaces the mean;
- the **MAD**, median absolute deviation - the median of `|x - median|` - replaces `sd`.

MAD is on a different scale to `sd`, so it is multiplied by **1.4826** (`ST.MAD_TO_SIGMA`), the
factor that makes the two agree *for Gaussian data*. That is the deal: after scaling, MAD tells
you what `sd` would have been **if the distribution were Gaussian and had no outliers**.

So the two together are a diagnostic, which is why `03` publishes both `R_sd` and `R_mad`. The
textbook reading is:

- **they agree** -> the noise is Gaussian and nothing exotic is happening;
- **`sd` exceeds `mad`** -> a few pixels are doing something the bulk is not, usually hot or
  telegraph pixels. The bulk width is `mad`; the difference is the outliers' contribution.

**And on this camera the textbook reading is wrong at low gain, in a way worth understanding.**
Look at `R_mad` in `bias_sweep.csv`: it is *exactly* 1.048356 at gain 0, 10, 20, 30 ... and 100
and 200. Identical to six digits across gains whose real read noise triples. That is not a
measurement, it is an artefact: `median(|d|)` on quantised data can only land on an integer, and
when the noise is around a count or two, it lands on **1** every time. 1.4826 x 1 / sqrt(2) =
1.0484, and MAD has stopped responding to the data at all.

Watch what that does to the ratio `sd / mad` as gain climbs. At gain 0 it is **0.63** - MAD
*over*-states the noise, because the grid holds it at 1.048 while the true width is 0.66. At gain
100 it is **1.36** - same pinned 1.048, but the true width has grown past it, so now MAD
*under*-states. Only above about gain 190 does MAD start responding to the data at all, and from
there the ratio settles into 1.03 to 1.38, where the excess is what it is supposed to be: real
outliers that `sd` counts and MAD rejects.

So below gain 190 the ratio is telling you about the ADC, and above it about the pixels. Same
two estimators, two completely different stories, and nothing in the number itself says which
one you are reading.

The practical rule for this project: **`R_sd` is the published number** - it is what actually
propagates into a stack - and `R_mad` is read as a *diagnostic of the diagnostic*, meaningful only
where the noise comfortably exceeds the quantisation grid, which here means above gain ~190.

In [ ]:
d = STACK[1] - STACK[0]
sd = d.std() / math.sqrt(2)
mad = ST.MAD_TO_SIGMA * np.median(np.abs(d - np.median(d))) / math.sqrt(2)
print(f"gain 100 pair difference:  sd {sd:.4f}   mad {mad:.4f}   ratio {sd / mad:.4f}")
print(f"  median(|d - median|) = {np.median(np.abs(d - np.median(d))):.1f} counts exactly "
      "-- the grid, not the noise")

spiked = d.copy()
rng = np.random.default_rng(2)
hit = rng.integers(0, spiked.size, 50)          # 50 pixels, 200 counts off
spiked.ravel()[hit] += 200.0
sd2 = spiked.std() / math.sqrt(2)
mad2 = ST.MAD_TO_SIGMA * np.median(np.abs(spiked - np.median(spiked))) / math.sqrt(2)
print(f"\nwith 50 spiked pixels:    sd {sd2:.4f}   mad {mad2:.4f}   ratio {sd2 / mad2:.4f}")
print(f"  {50 / d.size:.3%} of pixels moved sd by {sd2 / sd - 1:+.1%} "
      f"and mad by {mad2 / mad - 1:+.2%}.  That is the robustness the estimator is for.")

sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
at15 = sweep[sweep.offset == 15].sort_values("gain")
fig, ax = plt.subplots(1, 2, figsize=(9, 2.7))
ax[0].plot(at15.gain, at15.R_sd, ".-", ms=3, lw=0.8, label="R_sd")
ax[0].plot(at15.gain, at15.R_mad, ".-", ms=3, lw=0.8, label="R_mad")
ax[0].set(yscale="log", xlabel="gain", ylabel="ADC counts", title="the two estimators")
ax[0].legend()
ax[1].plot(at15.gain, at15.R_sd / at15.R_mad, ".-", ms=3, lw=0.8, color="crimson")
ax[1].axhline(1, color="0.6", lw=0.8)
ax[1].set(xlabel="gain", ylabel="R_sd / R_mad", title="below ~190 the MAD is pinned to the grid")
fig.tight_layout()

## 7. The noise, and how well we know the noise

These are different numbers and they are easy to conflate.

`R = 1.0768` counts is **the noise**: how much a pixel wobbles. It does not get smaller with more
frames, because it is a property of the camera.

`+/- 0.0005` counts is **how well we know that 1.0768**: the *standard error*. It gets smaller
with more frames, as `spread / sqrt(n)`, because it is a property of our measurement rather than
of the camera. Ten pairs per gain is enough to know `R` to about one part in 2,000 - `R_err` at
the HCG threshold is 0.00046 counts on a value of 1.0768 - which is why the constants file can
quote four decimals without embarrassment.

The rule of thumb worth memorising: **the uncertainty on a mean of `n` things is the spread of
those things divided by `sqrt(n)`.** Every `uncertainty` field in `results/` is some version of
this question - "how much would this number move if we shot the session again?" - and a constant
without one is a number nobody can argue with, which is worse than a number with a large error
bar.

In [ ]:
per_pair = np.array([(STACK[i + 1] - STACK[i]).std() / math.sqrt(2)
                     for i in range(0, 20, 2)])
print("R from each of 10 independent pairs, counts:")
print(np.round(per_pair, 4))
print(f"\nmean          {per_pair.mean():.4f}   <- the noise")
print(f"spread (sd)   {per_pair.std(ddof=1):.4f}   <- how much one pair's estimate wanders")
print(f"standard error {per_pair.std(ddof=1) / math.sqrt(len(per_pair)):.4f}   "
      f"<- how well we know the mean, = spread / sqrt({len(per_pair)})")

n = np.arange(1, len(per_pair) + 1)
fig, ax = plt.subplots(figsize=(4.2, 2.6))
ax.plot(n, [per_pair[:k].std(ddof=1) / math.sqrt(k) if k > 1 else np.nan for k in n],
        "o-", ms=3, color="0.3", label="standard error")
ax.axhline(per_pair.std(ddof=1), ls="--", color="crimson", label="spread of one pair")
ax.set(xlabel="pairs used", ylabel="counts", title="the noise stays; the uncertainty shrinks")
ax.legend(); fig.tight_layout()

## 8. Quantisation: the grid underneath every number

The ADC does not return a continuous value. It returns an integer, and this project's unit is
one such integer - the ADC count (`CLAUDE.md`'s units rule; the stored FITS value is 16x it).

When the noise is much wider than the grid, quantisation is invisible. When the noise is
*comparable* to the grid - and at gain 0 this camera's read noise is 0.66 counts, well under one
step - two things happen that will otherwise confuse you:

- The histogram is a picket fence, not a curve. A "distribution" of two or three spikes still has
  a perfectly well-defined `sd`, and that `sd` is partly a statement about the grid.
- **Differences of medians collapse to zero.** If the median dark and the median bias land on the
  same integer, their difference is exactly 0 - not because there is no dark current, but because
  the grid swallowed it. This is L14's negative-dark-current story, and the reason the dark
  measurement uses means over millions of pixels rather than medians.

The practical rule: at low gain, prefer means (which average the grid away over many pixels) and
distrust any statistic that depends on individual values being distinguishable.

In [ ]:
lo = plane("coarse_g000_o015_000")          # gain 0: sd well under one count
hi = plane("coarse_g300_o015_000")          # gain 300: sd of several counts

fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
for a, p, g in zip(ax, (lo, hi), (0, 300)):
    lo_e, hi_e = np.floor(p.min()) - 0.5, np.ceil(p.max()) + 1.5
    a.hist(p.ravel(), bins=np.arange(lo_e, hi_e, 1.0), color="0.4", edgecolor="white")
    a.set(title=f"gain {g}: sd {p.std():.2f} counts", xlabel="ADC counts")
    a.set_xlim(p.mean() - 6 * max(p.std(), 1), p.mean() + 6 * max(p.std(), 1))
fig.tight_layout()

print(f"gain 0:   {len(np.unique(lo)):>4} distinct values in the whole plane")
print(f"gain 300: {len(np.unique(hi)):>4} distinct values")
print(f"\ngain 0 median {np.median(lo):.1f}, mean {lo.mean():.4f} -- the median is an integer\n"
      f"and always will be; the mean sees the sub-count detail because {NPX:,} pixels\n"
      "split across a few codes encode it in their *proportions*.")

## 9. The Gaussian tail, and what "15 read noises of headroom" buys

Noise that is the sum of many small independent contributions comes out bell-shaped - Gaussian -
and that shape has a known, very steep tail. The fraction of values further than `k` sigma below
the centre is what decides whether a distribution sitting `k` sigma above zero gets truncated by
the floor.

The numbers are worth internalising once, because they explain why the offset argument looks the
way it does:

| distance | fraction below it | on one 512 x 512 plane (262,144 px) |
|---|---|---|
| 3 sigma | 1.3e-3 | ~354 pixels |
| 4 sigma | 3.2e-5 | ~8 pixels |
| 5 sigma | 2.9e-7 | ~0.08 pixels |
| 10 sigma | 7.6e-24 | never |
| 15 sigma | 3.7e-51 | never, by an enormous margin |

So `15 R` of headroom is not a cautious margin - it is a **guarantee**, under the Gaussian
assumption, that the floor cannot touch the distribution. Which immediately raises the right
question: is the assumption true? The comparison the project actually publishes is the *measured*
zero fraction against this predicted one. At offset 0, gain 0 the measured fraction is **1.7x**
what a Gaussian of that width predicts - so the real distribution has a fatter low tail than the
bell curve, and the honest reading is "the Gaussian tail is a lower bound on the trouble".

In [ ]:
def tail(k):
    '''Fraction of a Gaussian lying more than k sigma below the centre.'''
    return 0.5 * math.erfc(k / math.sqrt(2))


px = NPX
print(f"{'k sigma':>8} {'fraction':>12} {f'pixels of {px:,}':>18}")
for k in (2, 3, 4, 5, 6, 10, 15):
    print(f"{k:>8} {tail(k):>12.2e} {tail(k) * px:>18.3g}")

sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
worst = sweep[(sweep.offset == 0)].nsmallest(1, "sigmas_above_zero").iloc[0]
print(f"\nthe measured case -- offset 0, gain {int(worst.gain)}:")
print(f"  pedestal {worst.pedestal:.2f} counts, R {worst.R_at_offset:.3f}, "
      f"{worst.sigmas_above_zero:.1f} R above the floor")
print(f"  Gaussian predicts {worst.zero_frac_gaussian:.2e} of pixels below zero")
print(f"  measured           {worst.zero_frac:.2e}   "
      f"({worst.zero_frac / worst.zero_frac_gaussian:.1f}x the prediction)")

## 10. Fractions are estimators; extremes are not

"The darkest pixel in this frame reads 0, so the offset is clipping" is the single most
seductive wrong argument in this subject, and L13 records a retired project acting on it.

The reason it is wrong is that **the minimum of a large sample is not a stable quantity, and it
grows with sample size for no physical reason at all.** Draw pure Gaussian noise and the darkest
of 1,000 draws sits about 3.0 sigma below centre; of 262,144, about 4.9; of ten million, about
5.4. (The textbook scaling `sqrt(2 ln n)` is the right shape and overshoots a little at these
sizes - it is an asymptotic result.)

So on one 512 x 512 plane you should *expect* the darkest pixel about 4.9 sigma below the
pedestal from noise alone. If the pedestal sits 4.2 R above zero, as it does at offset 0 and
gain 0, the minimum was always going to hit the floor - and would have hit it on a perfect
sensor. The minimum tells you almost nothing about the distribution, and what little it tells
you changes with how many pixels you looked at.

A **fraction** - what proportion of pixels are at the floor - is an estimator: it converges as
you take more data, it has a meaningful uncertainty, and it can be compared against a prediction
(section 9). That is why the criterion is 0.1% of pixels, not "no pixels".

And then the refinement that the sweep forced, which is worth reading as a lesson in what a
threshold cannot do on its own. **A fraction still cannot tell you *why* a pixel reads zero.**
Two causes look identical in the count:

- **clipping** - the distribution genuinely reaches the floor, so the fraction tracks the
  headroom and dies when you raise the offset;
- **telegraph pixels** - a handful of pixels that randomly switch low regardless of headroom, so
  the fraction is indifferent to the offset entirely.

The test that separates them needs pixels, not statistics: **are they the same pixels?** Dead
pixels are; telegraph pixels are not. Section 6 of `04` runs that test on the gain-600 frames.

In [ ]:
rng = np.random.default_rng(3)
for n in (1_000, 100_000, NPX, 10_000_000):
    mins = [rng.normal(0, 1, n).min() for _ in range(5)]
    tag = "  <- one plane of this ROI" if n == NPX else ""
    print(f"minimum of {n:>10,} Gaussian draws: {np.mean(mins):+.2f} sigma   "
          f"(asymptotic {-math.sqrt(2 * math.log(n)):+.2f}){tag}")
print("\nThe extreme moves by two sigma across those rows.  Nothing about the sensor\n"
      "changed between them -- only how many pixels were looked at.")

# The fraction, by contrast, is stable: same distribution, different sample sizes.
print("\nthe fraction below -3 sigma, same draws:")
for n in (1_000, 100_000, NPX, 10_000_000):
    f = [np.mean(rng.normal(0, 1, n) < -3) for _ in range(5)]
    print(f"  n = {n:>10,}: {np.mean(f):.2e}  (true value 1.35e-03)")

## 11. Fitting a line, and reading the residuals

A **fit** finds the straight line that comes closest to a set of points, "closest" meaning the
sum of squared vertical distances is as small as possible (`np.polyfit`). The distances
themselves are the **residuals**, and they are where all the information is: the fit coefficients
tell you what the model says, the residuals tell you whether the model is any good.

Two habits this project keeps:

- **Look at residuals as a picture, not as a single number.** Random scatter about zero means the
  line is the right shape. A residual pattern that *curves* means it is not, no matter how small
  the numbers are.
- **Quote the worst residual, not the average one, when the claim is about being misled.** The
  pedestal fit's uncertainty field carries the maximum residual, because a pedestal is subtracted
  from every frame and the question a user has is "how wrong can this be", not "how wrong is it
  on average".

The pedestal is the worked example. The model is `pedestal = A + B * amplification`, where
`amplification = 10 ** (gain / 200)` because ZWO's gain unit is 0.1 dB - so gain 200 is 20 dB, a
factor of 10. `A` is the digital offset, added after the amplifier - section 0 measured it
directly, at 4.000 counts per offset unit and the same at every gain; `B * amplification` is the
analogue part that gets amplified along with everything else.

Fitted **once across all gains** it looks respectable and is wrong in a specific, structured way,
because the sensor changes conversion gain at 200 and the two halves genuinely have different
`B`. Fitted **per branch** the residuals collapse. That is section 3 of `04`, and the residual
plot below is the entire argument for splitting.

In [ ]:
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
at15 = sweep[sweep.offset == 15].sort_values("gain")
amp = 10 ** (at15.gain / 200.0)

fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
for a, (label, sel) in zip(ax, [("one fit, all gains", np.ones(len(at15), bool)),
                                ("split at gain 200", None)]):
    if sel is not None:
        B, A = np.polyfit(amp, at15.pedestal, 1)
        resid = at15.pedestal - (A + B * amp)
        a.plot(at15.gain, resid, "o", ms=2.5, color="crimson")
    else:
        for name, mask in (("lcg", at15.gain < 200), ("hcg", at15.gain >= 200)):
            B, A = np.polyfit(amp[mask], at15.pedestal[mask], 1)
            r = at15.pedestal[mask] - (A + B * amp[mask])
            a.plot(at15.gain[mask], r, "o", ms=2.5, label=f"{name}: A={A:.1f} B={B:.3f}")
        a.legend(fontsize=6)
    a.axhline(0, color="0.6", lw=0.8)
    a.set(title=label, xlabel="gain", ylabel="residual, ADC counts")
fig.tight_layout()
print("Left: the residuals are not scatter -- they are a shape, and a shape in the residuals\n"
      "means the model is wrong.  Right: same data, one extra parameter, structure gone.")

## 12. A slope against its own uncertainty

The last idea, and the one that turns a measurement into a decision.

Fit a line to the pedestal against time and you always get a slope. It is never exactly zero -
noise guarantees that. So "the pedestal drifts at -0.00133 counts per minute" is not yet a
statement about the camera; it becomes one only when compared against **how much that slope
would wander if you shot the trace again**.

That number is the **standard error of the slope**, and it is a different thing from the scatter
of the points about the line. The scatter here is 0.254 counts; the slope's own uncertainty is
0.0028 counts/min, about ninety times smaller, because 450 points spread over 15 minutes pin a
line down far better than any single point is known. The comparison that decides the question is
slope against *slope* error:

`-0.00133 / 0.00277 = -0.48`

Under half of one error bar. The correct reading is not "a tiny drift" but **"no drift detected,
at a sensitivity of about 0.006 counts/min"**, and the sensitivity is the useful output.

**A note on the published constant.** `pedestal_drift_rate` in `bias_constants.json` carries
0.254 in its `uncertainty` field - the residual scatter, not the slope error. That is
conservative rather than wrong, and the conclusion it was used for is the same conclusion,
but the two numbers answer different questions and only one of them is the uncertainty on the
value being published.

That distinction is the difference between the two write-ups this project could have published:

- *"the pedestal falls by 0.02 counts over 15 minutes"* - technically the fitted value, and
  meaningless;
- *"no drift above 0.25 counts over 15 minutes, so interleaving bias frames between darks is a
  precaution rather than a requirement"* - which is what session 02's design actually rests on.

**An upper bound is a result.** It is quoted as one, with the bound stated, and it is exactly
what L14 demands for dark current too.

In [ ]:
drift = pd.read_csv(RESULTS / "pedestal_drift.csv")
slope, intercept = np.polyfit(drift.elapsed_s, drift.pedestal, 1)
resid = drift.pedestal - (slope * drift.elapsed_s + intercept)
span = slope * drift.elapsed_s.max()

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.plot(drift.elapsed_s / 60, drift.pedestal, ".", ms=2, color="0.5")
ax.plot(drift.elapsed_s / 60, slope * drift.elapsed_s + intercept, color="crimson", lw=1,
        label=f"fit: {slope * 60:+.4f} counts/min")
ax.set(xlabel="minutes", ylabel="pedestal, ADC counts", title="the drift trace, gain 100")
ax.legend(); fig.tight_layout()

n = len(drift)
se = resid.std(ddof=2) / (drift.elapsed_s.std(ddof=1) * math.sqrt(n - 1))

print(f"{n} frames over {drift.elapsed_s.max() / 60:.1f} minutes")
print(f"fitted slope                {slope * 60:+.5f} counts/min")
print(f"standard error of the slope  {se * 60:.5f} counts/min")
print(f"slope / its error           {slope / se:+.2f}   <- the number that decides it")
print(f"\nscatter of points about the line {resid.std():.3f} counts "
      "(a different question, and 90x larger)")
print(f"fitted change over the whole trace {span:+.3f} counts")
print(f"\n2-sigma bound on any drift: |rate| < {2 * se * 60:.4f} counts/min, "
      f"i.e. under {2 * se * 60 * 15:.3f} counts in 15 minutes.")
print("That bound is the result.  'The pedestal falls at 0.00133 counts/min' is not.")

---

## What to take away

0. Where a number comes from decides how it behaves: the gain multiplies every
   noise source upstream of it and none of the ones downstream.
1. Everything is a distribution. Ask *centre or width* before anything else.
2. Ask *across pixels or across frames*. They answer different questions.
3. Variances add. Every `sqrt(2)` and `sqrt(N)` in this repo is that one fact.
4. Difference two adjacent frames to measure noise; never a single frame, never an extrapolation.
5. `sd` and `mad` disagreeing is information, not a problem to be tidied.
6. The noise and the uncertainty on the noise are different numbers with different behaviour.
7. Near the quantisation grid, prefer means and distrust medians.
8. Compare a measured fraction against its predicted tail; that is what makes it evidence.
9. Never argue from an extreme value.
10. Read residuals as a shape.
11. A slope inside its own error bar is a bound, and a bound is a result.

The next notebook, `04`, uses every one of these on session 01's numbers.